# 🧠 LSTM vs GRU
**Course:** Deep Learning  
**Lecture:** 5 — Recurrent Neural Networks (RNNs)  
**Part:** b — LSTM vs GRU

---

> **Where we are:** Now that we know Vanilla RNNs struggle with long sequences due to vanishing gradients, we introduce Gated units (LSTM and GRU) which solve this by selectively passing information.

**What you'll learn:**
- LSTM mechanics: Forget, Input, Output gates, and the Cell State
- GRU mechanics: Reset and Update gates
- Implementing gates from scratch
- Comparing LSTM and GRU in parameters and performance

**Exam relevance:** ⭐⭐⭐ (Knowing the gates and difference between LSTM/GRU is heavily tested)


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import time
import os
import pandas as pd

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12


## 📖 1. The LSTM Cell (Long Short-Term Memory)

The LSTM uses an internal cell state $C_t$ ("memory highway") and 3 gates to control information flow.
- **Forget gate:** $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ — what to forget from the past cell state
- **Input gate:** $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ — what new info to store
- **Cell state update:** $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$
  - Where candidate $\tilde{C}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$
- **Output gate:** $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ — what to output
- **Hidden state:** $h_t = o_t \odot \tanh(C_t)$


## 📖 2. The GRU Cell (Gated Recurrent Unit)

GRU is a simplified LSTM with fewer parameters (2 gates instead of 3, no separate cell state).
- **Reset gate:** $r_t = \sigma(W_r [h_{t-1}, x_t])$ — how much past to forget
- **Update gate:** $z_t = \sigma(W_z [h_{t-1}, x_t])$ — how much to update (acts like forget + input)
- **New hidden:** $h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$
  - Where candidate $\tilde{h}_t = \tanh(W_h [r_t \odot h_{t-1}, x_t])$


## 🔀 Your Options: Gated RNNs

At this step, you can choose between:

| Option | Pros | Cons | When to use |
|---|---|---|---|
| LSTM | High expressive power | More parameters, slower | Complex sequence tasks |
| GRU | Fewer params, faster training | Slightly less expressive | Good default, less data |


In [ ]:
# LSTM gates - all use sigmoid except the cell candidate which uses tanh
f_t = sigmoid(concat @ W_f)                      # forget gate: what to erase from cell state
i_t = sigmoid(concat @ W_i)                      # input gate: what new info to store
C_tilde = np.tanh(concat @ W_C)                  # cell candidate: new info to add
C_t = f_t * C_prev + i_t * C_tilde              # cell state: forget old + add new
o_t = sigmoid(concat @ W_o)                      # output gate: what to expose from cell state
h_t = o_t * np.tanh(C_t)                        # hidden state: filtered view of cell state

print('LSTM one-step output:')
print(f'  forget gate f_t:      {f_t.round(3)}  (close to 1 = keep, 0 = forget)')
print(f'  input gate  i_t:      {i_t.round(3)}')
print(f'  cell state  C_t:      {C_t.round(3)}  (the long-term memory)')
print(f'  output gate o_t:      {o_t.round(3)}')
print(f'  hidden state h_t:     {h_t.round(3)}')

# GRU (fewer parameters - only 2 gates)
W_r_gru = np.random.randn(8, 4)  # reset gate weights
W_z_gru = np.random.randn(8, 4)  # update gate weights
W_h_gru = np.random.randn(8, 4)  # candidate hidden state weights

r_t = sigmoid(concat @ W_r_gru)                  # reset gate: how much past to forget
z_t = sigmoid(concat @ W_z_gru)                  # update gate: how much to update
# candidate: uses reset gate to mask past hidden state
concat_reset = np.concatenate((r_t * h_prev, x_t))
h_tilde = np.tanh(concat_reset @ W_h_gru)       # candidate new hidden state
h_t_gru = (1 - z_t) * h_prev + z_t * h_tilde   # interpolate: keep old or use new

print('\nGRU one-step output:')
print(f'  reset gate  r_t:      {r_t.round(3)}')
print(f'  update gate z_t:      {z_t.round(3)}')
print(f'  new hidden  h_t:      {h_t_gru.round(3)}')
print('\nGRU uses 2 gates vs LSTM 4 gates -> fewer parameters, often similar accuracy.')

In [ ]:
# ✅ Solution: Implement gates conceptually

f_t = sigmoid(np.dot(concat, W_f))
i_t = sigmoid(np.dot(concat, W_i))
C_tilde = np.tanh(np.dot(concat, W_C))

# Cell state update
C_t = f_t * C_prev + i_t * C_tilde

o_t = sigmoid(np.dot(concat, W_o))
h_t = o_t * np.tanh(C_t)

print("LSTM output h_t:", h_t)


In [ ]:
# 🔬 Comparison: Parameter Counts and Training

units = 64

# Parameter counting
model_rnn = tf.keras.Sequential([tf.keras.layers.SimpleRNN(units, input_shape=(10, 1))])
model_gru = tf.keras.Sequential([tf.keras.layers.GRU(units, input_shape=(10, 1))])
model_lstm = tf.keras.Sequential([tf.keras.layers.LSTM(units, input_shape=(10, 1))])

print(f"SimpleRNN Params: {model_rnn.count_params()}")
print(f"GRU Params:       {model_gru.count_params()}")
print(f"LSTM Params:      {model_lstm.count_params()}")

# Compare training on synthetic data (Epileptic Seizure placeholder)
X_train = np.random.randn(1000, 10, 1)
y_train = np.random.randint(0, 2, 1000)

models = {'SimpleRNN': model_rnn, 'GRU': model_gru, 'LSTM': model_lstm}
results = {}

for name, model in models.items():
    m = tf.keras.Sequential([
        model,
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    start = time.time()
    hist = m.fit(X_train, y_train, epochs=10, validation_split=0.2, verbose=0, batch_size=32)
    t = time.time() - start
    
    results[name] = {
        'val_acc': hist.history['val_accuracy'],
        'time': t
    }

# Plot training curves
plt.figure(figsize=(10, 5))
for name, res in results.items():
    plt.plot(res['val_acc'], label=f"{name} (Time: {res['time']:.1f}s)")
plt.title('Validation Accuracy: SimpleRNN vs GRU vs LSTM')
plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.show()

# Table output
print("\n| Model | # Params | Train Time (10 epochs) |")
print("|---|---|---|")
for name in models:
    print(f"| {name} | {models[name].count_params()} | {results[name]['time']:.2f}s |")



## 🎓 Exam Corner

### Question 1 (Conceptual)
The GRU has 2 gates, the LSTM has 4. Name all of them. What is the purpose of the forget gate in LSTM?

<details><summary>💡 Answer</summary>

**GRU gates:** Reset gate, Update gate.
**LSTM gates:** Forget gate, Input gate, Output gate (and the candidate cell state is sometimes called the 4th).
**Forget gate purpose:** It controls what information from the previous cell state $C_{t-1}$ should be discarded or kept.
</details>

### Question 2 (Practical)
An LSTM with 128 units and input size 10: how many parameters does it have?

<details><summary>💡 Answer</summary>

For one gate, weights = `(input_size + units) * units` and biases = `units`.
Total per gate = `(10 + 128) * 128 + 128 = 138 * 128 + 128 = 17664 + 128 = 17792`.
Since LSTM has 4 neural network layers (3 gates + 1 candidate), total = `4 * 17792 = 71168` parameters.
</details>
